In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd

In [ ]:
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
X_val = pd.read_csv("X_val.csv")

In [ ]:
smiles1_train = X_train["Smiles 1"].tolist()
smiles2_train = X_train["Smiles 2"].tolist()

smiles1_val = X_val["Smiles 1"].tolist()
smiles2_val = X_val["Smiles 2"].tolist()

smiles1_test = X_test["Smiles 1"].tolist()
smiles2_test = X_test["Smiles 2"].tolist()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("UdS-LSV/smole-bert")
model = AutoModel.from_pretrained("UdS-LSV/smole-bert")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def get_embeddings(smiles_list, batch_size=1024):
    all_embeddings = []
    n_batches = len(smiles_list) // batch_size + 1
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]
        encoded = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128)
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            output = model(**encoded)
        # CLS token embedding as molecular representation
        embeddings = output.last_hidden_state[:, 0, :]
        all_embeddings.append(embeddings.cpu().numpy())
        if (i // batch_size + 1) % 10 == 0:
            print(f"Batch {i // batch_size + 1}/{n_batches}")
    return np.concatenate(all_embeddings, axis=0)

# Compute and save
E1_train = get_embeddings(smiles1_train)
E2_train = get_embeddings(smiles2_train)

E1_val = get_embeddings(smiles1_val)
E2_val = get_embeddings(smiles2_val)

E1_test = get_embeddings(smiles1_test)
E2_test = get_embeddings(smiles2_test)

np.save("E1_train.npy", E1_train)
np.save("E2_train.npy", E2_train)

np.save("E1_val.npy", E1_val)
np.save("E2_val.npy", E2_val)

np.save("E1_test.npy", E1_test)
np.save("E2_test.npy", E2_test)

ModuleNotFoundError: No module named 'smiles_featurizers'